In [19]:
import torch
from transformer_lens import HookedTransformer
from sae_lens import SAE, ActivationsStore
from datasets import load_dataset

device = "cuda:4" if torch.cuda.is_available() else "cpu"

In [20]:
# ── 1. Load model ─────────────────────────────────────────────
model = HookedTransformer.from_pretrained("gemma-2-2b", device=device)
model.eval()

Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 55.38it/s]


Loaded pretrained model gemma-2-2b into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-25): 26 x TransformerBlock(
      (ln1): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln1_post): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2_post): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): GatedMLP(
        (hook_pre): HookPoint()
        (hook_pre_linear): HookPoint()
        (

In [31]:
# # ── 2. Load your trained SAE from checkpoint ──────────────────
# sae, cfg_dict, log_sparsity = SAE.from_pretrained(
#     release="suchitg/selfconsuming-saes",                        
#     sae_id="gen_000/blocks.4.hook_resid_post",
#     device=device,
# )
# sae.eval()

############################ OR #################################

# # ── 2. Load gemmascope SAE ──────────────────
sae, cfg_dict, log_sparsity = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res",
    sae_id="layer_4/width_16k/average_l0_31",
    device=device,
)
sae.eval()

# Sanity check before running eval
print(f"Architecture:  {cfg_dict.get('architecture')}")
print(f"Hook name:     {cfg_dict.get('hook_name')}")
print(f"d_in:          {cfg_dict.get('d_in')}")
print(f"Normalize:     {cfg_dict.get('normalize_activations')}")
print(f"dtype:         {cfg_dict.get('dtype')}")

Architecture:  jumprelu
Hook name:     None
d_in:          2304
Normalize:     none
dtype:         float32


/tmp/ipykernel_1662997/4144235213.py:12: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, log_sparsity = SAE.from_pretrained(


In [32]:
# ── 3. Quick reconstruction test on a single forward pass ─────
tokens = model.to_tokens("The Eiffel Tower is located in")

with torch.no_grad():
    _, cache = model.run_with_cache(tokens, names_filter="blocks.4.hook_resid_post")
    acts = cache["blocks.4.hook_resid_post"]  # (batch, seq, d_model)
    acts_flat = acts.reshape(-1, acts.shape[-1])  # (batch*seq, d_model)

    feature_acts = sae.encode(acts_flat)
    recon = sae.decode(feature_acts)

    # L0: mean active features per token
    l0 = (feature_acts > 0).float().sum(-1).mean()
    # MSE
    mse = (acts_flat - recon).pow(2).mean()
    # Explained variance
    total_var = (acts_flat - acts_flat.mean(0)).pow(2).mean()
    recon_var = (acts_flat - recon).pow(2).mean()
    explained_var = 1 - recon_var / total_var

    print(f"L0:                {l0:.1f}")
    print(f"MSE:               {mse:.4f}")
    print(f"Explained variance:{explained_var:.4f}")

L0:                672.3
MSE:               362.8779
Explained variance:-9.2623


In [33]:
# See what's in the config
print(cfg_dict)

# Specifically look for scaling factor
print(f"scaling_factor: {cfg_dict.get('scaling_factor')}")
print(f"normalize_activations: {cfg_dict.get('normalize_activations')}")

{'d_in': 2304, 'd_sae': 16384, 'dtype': 'float32', 'device': 'cuda:4', 'apply_b_dec_to_input': False, 'normalize_activations': 'none', 'reshape_activations': 'none', 'metadata': {'sae_lens_version': '6.36.2', 'sae_lens_training_version': None, 'model_name': 'gemma-2-2b', 'hook_name': 'blocks.4.hook_resid_post', 'hook_head_index': None, 'prepend_bos': True, 'dataset_path': 'monology/pile-uncopyrighted', 'context_size': 1024, 'neuronpedia_id': None}, 'architecture': 'jumprelu'}
scaling_factor: None
normalize_activations: none


In [34]:
# ── Find the correct scaling factor ───────────────────────────
# The scaling factor = 1 / sqrt(mean_sq_norm of raw activations)
# Estimate it from a batch of real activations

@torch.no_grad()
def estimate_scaling_factor(model, hook_name, dataset_path, n_batches=50, device="cuda"):
    from datasets import load_dataset
    dataset = load_dataset(dataset_path, streaming=True, split="train", trust_remote_code=True)
    
    sq_norms = []
    for i, sample in enumerate(dataset):
        if i >= n_batches:
            break
        tokens = model.to_tokens(sample["text"])[:, :512]
        _, cache = model.run_with_cache(tokens, names_filter=hook_name)
        acts = cache[hook_name].reshape(-1, model.cfg.d_model)
        sq_norms.append(acts.pow(2).mean().item())
    
    mean_sq_norm = torch.tensor(sq_norms).mean()
    scaling_factor = 1.0 / mean_sq_norm.sqrt()
    print(f"Mean sq norm of activations: {mean_sq_norm:.2f}")
    print(f"Scaling factor (1/sqrt(MSN)): {scaling_factor:.6f}")
    return scaling_factor

scaling_factor = estimate_scaling_factor(
    model,
    hook_name="blocks.4.hook_resid_post",
    dataset_path="monology/pile-uncopyrighted",
    device=device,
)

Mean sq norm of activations: 4.42
Scaling factor (1/sqrt(MSN)): 0.475598


In [35]:
@torch.no_grad()
def get_metrics(model, sae, hook_name, prompt, scaling_factor):
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens, names_filter=hook_name)
    acts = cache[hook_name].reshape(-1, sae.cfg.d_in)  # raw activations

    # Normalize in, encode, decode, un-normalize out
    acts_normed  = acts * scaling_factor
    feature_acts = sae.encode(acts_normed)
    recon_normed = sae.decode(feature_acts)
    recon        = recon_normed / scaling_factor        # back to original scale

    l0  = (feature_acts > 0).float().sum(-1).mean()
    mse = (acts - recon).pow(2).mean()
    ev  = 1 - (acts - recon).pow(2).mean() / acts.var()

    print(f"L0:                 {l0:.1f}")
    print(f"MSE:                {mse:.4f}")
    print(f"Explained variance: {ev:.4f}")
    return feature_acts, recon

@torch.no_grad()
def ce_delta(model, sae, hook_name, prompt, scaling_factor):
    tokens = model.to_tokens(prompt)

    def sae_hook(acts, hook):
        acts_normed  = acts * scaling_factor
        feature_acts = sae.encode(acts_normed)
        recon_normed = sae.decode(feature_acts)
        return recon_normed / scaling_factor

    baseline = model(tokens, return_type="loss")
    spliced  = model.run_with_hooks(
        tokens,
        return_type="loss",
        fwd_hooks=[(hook_name, sae_hook)],
    )
    print(f"Baseline CE: {baseline:.4f}")
    print(f"Spliced CE:  {spliced:.4f}")
    print(f"Delta CE:    {spliced - baseline:.4f}")

prompt = "The Eiffel Tower is located in"
get_metrics(model, sae, "blocks.4.hook_resid_post", prompt, scaling_factor)
ce_delta(model, sae, "blocks.4.hook_resid_post", prompt, scaling_factor)

L0:                 267.6
MSE:                113.5498
Explained variance: -1.4437
Baseline CE: 4.0867
Spliced CE:  11.9316
Delta CE:    7.8449


In [23]:
# ── 4. CE loss degradation (the key quality metric) ───────────
# Splice the SAE into the model and measure how much worse
# the LM loss gets. <0.05 nats degradation is good.

def replacement_hook(acts, hook):
    feature_acts = sae.encode(acts)
    return sae.decode(feature_acts)

tokens = model.to_tokens("The quick brown fox jumps over the lazy dog")
labels = tokens[:, 1:]  # shift for next-token prediction

with torch.no_grad():
    # Baseline LM loss
    baseline_loss = model(tokens, return_type="loss")

    # SAE-spliced loss
    spliced_loss = model.run_with_hooks(
        tokens,
        return_type="loss",
        fwd_hooks=[("blocks.4.hook_resid_post", replacement_hook)],
    )

delta_ce = spliced_loss - baseline_loss
print(f"Baseline CE loss:  {baseline_loss:.4f}")
print(f"SAE-spliced loss:  {spliced_loss:.4f}")
print(f"Delta CE (↓ better):{delta_ce:.4f}")

Baseline CE loss:  2.3513
SAE-spliced loss:  2.4009
Delta CE (↓ better):0.0496


In [24]:
# ── 5. Dead feature count ─────────────────────────────────────
store = ActivationsStore.from_sae(
    model=model,
    sae=sae,
    dataset="monology/pile-uncopyrighted",
    dataset_trust_remote_code=True,
    streaming=True,
    store_batch_size_prompts=8,
    train_batch_size_tokens=4096,
    n_batches_in_buffer=4,
    device=device,
)

n_batches_to_check = 100
feature_counts = torch.zeros(sae.cfg.d_sae, device=device)

with torch.no_grad():
    for _ in range(n_batches_to_check):
        acts = store.next_batch()
        feature_acts = sae.encode(acts)
        feature_counts += (feature_acts > 0).float().sum(0)

dead_features = (feature_counts == 0).sum().item()
print(f"Dead features: {dead_features} / {sae.cfg.d_sae} "
      f"({100*dead_features/sae.cfg.d_sae:.1f}%)")

Dead features: 0 / 16384 (0.0%)
